# Práctica — Clase 8. Gradientes de política y métodos Actor-Crítico

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En esta práctica programas tú misma o tú mismo el corazón de REINFORCE: el cálculo de los retornos descontados y la pérdida del gradiente de política. Después, añades una línea base y compruebas en primera persona cuánto ayuda a reducir la varianza del entrenamiento.

Usas el mismo entorno de la clase: CartPole-v1, donde un carro debe mantener un palo en equilibrio moviéndose a izquierda o derecha.


## Preparación

**Ejecuta las celdas siguientes en orden.** Instalan gymnasium, importan las librerías y te dejan listas dos piezas ya completas y funcionando: la red de la política y la recolección de un episodio. No hace falta que las edites.

In [ ]:
# Instala gymnasium (PyTorch ya viene preinstalado en Colab)
!pip install gymnasium

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

### La red de la política (ya completa)

`RedPolitica` representa `π_θ(a|s)`: recibe el estado de CartPole y devuelve una distribución de probabilidad sobre sus dos acciones, empujar a la izquierda o a la derecha. Es la misma red que viste en la clase.

In [ ]:
class RedPolitica(nn.Module):
    """Red neuronal pequeña que representa la política π_θ(a|s). Devuelve una
    distribución de probabilidad sobre las dos acciones posibles."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, estado):
        logits = self.red(estado)
        return torch.softmax(logits, dim=-1)

### Recolectar un episodio (ya completa)

`ejecutar_episodio` sigue la política actual desde el reinicio del entorno hasta que termina el episodio, y guarda la log-probabilidad de cada acción tomada (la necesitas para el gradiente de política) junto con la recompensa de cada paso. Es la misma función de la clase.

In [ ]:
def ejecutar_episodio(env, politica):
    """Recolecta un episodio completo siguiendo la política actual. Devuelve
    la log-probabilidad de cada acción tomada y la recompensa de cada paso."""
    log_probs = []
    recompensas = []
    estado, _ = env.reset()
    terminado = False
    truncado = False

    while not (terminado or truncado):
        estado_tensor = torch.FloatTensor(estado).unsqueeze(0)
        probs = politica(estado_tensor)
        distribucion = torch.distributions.Categorical(probs)
        accion = distribucion.sample()

        log_probs.append(distribucion.log_prob(accion))
        estado, recompensa, terminado, truncado, _ = env.step(accion.item())
        recompensas.append(recompensa)

    return log_probs, recompensas

## Ejercicio 1 — Completa los retornos y la pérdida de REINFORCE

En la clase viste que REINFORCE necesita dos ingredientes por episodio: el retorno de cada paso (la suma descontada de recompensas futuras) y la pérdida que empuja hacia arriba la probabilidad de las acciones que salieron bien.

Abajo tienes `calcular_retornos` y `entrenar_reinforce` con dos `# TODO`. El resto del bucle de entrenamiento —crear el entorno, la política, el optimizador, ejecutar episodios y actualizar los parámetros— ya está construido.

**Tu tarea es completar los dos `# TODO`:**

1. En `calcular_retornos`, dentro del bucle `for t in reversed(...)`, acumula el retorno con descuento: `G = recompensas[t] + gamma * G`, reutilizando el `G` que ya calculaste en el paso siguiente
2. En `entrenar_reinforce`, dentro del bucle `for log_prob, G in zip(...)`, calcula el término de la pérdida para ese paso: `-log_prob * G` (el signo negativo convierte el ascenso de gradiente que quieres en el descenso que usa PyTorch)

**Pistas**

- La fórmula de `calcular_retornos` es exactamente la que viste en la clase: `G_t = r_t + gamma * G_{t+1}`, calculada de atrás hacia adelante
- La pérdida completa es `-sum(log_prob(a_t|s_t) * G_t)`: aquí solo te falta el término de un paso, la suma ya está hecha con `torch.stack(...).sum()`
- Con los dos `# TODO` sin completar, la celda de prueba de abajo se ejecuta sin ningún error, pero la recompensa media no mejora: sigue rondando entre 20 y 30, la recompensa típica de una política que actúa al azar. Esa es tu señal de que aún te falta completar algo


In [ ]:
def calcular_retornos(recompensas, gamma=0.99):
    """Calcula el retorno 'reward-to-go' en cada paso del episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Se recorre la lista de atrás hacia adelante para reutilizar el cálculo
    del paso siguiente en cada paso anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        # TODO: acumula el retorno con descuento, reutilizando el G del paso
        # siguiente: G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_reinforce(n_episodios, gamma=0.99, lr=1e-2, seed=0, verbose=True):
    """Bucle de entrenamiento de REINFORCE. Ya está completo: solo te falta
    terminar calcular_retornos (arriba) y el término de la pérdida (abajo)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)  # sembramos el entorno una vez, para que el experimento sea reproducible
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = calcular_retornos(recompensas, gamma)
        retornos_tensor = torch.FloatTensor(retornos)

        terminos_perdida = []
        for log_prob, G in zip(log_probs, retornos_tensor):
            # TODO: sustituye la siguiente línea por el término correcto de
            # la pérdida de REINFORCE para este paso: -log_prob * G
            termino = log_prob * 0.0
            terminos_perdida.append(termino)
        perdida = torch.stack(terminos_perdida).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas

**Ejecuta la celda siguiente para probar tu implementación.** Compara la recompensa media de los primeros 20 episodios contra los últimos 20: si tu código funciona, la segunda debería ser bastantes veces mayor que la primera.

In [ ]:
historial_prueba = entrenar_reinforce(n_episodios=250, seed=0)

print("\nRecompensa media, primeros 20 episodios:", np.mean(historial_prueba[:20]))
print("Recompensa media, últimos 20 episodios:", np.mean(historial_prueba[-20:]))

<details>
<summary>💡 Ver solución</summary>

```python
def calcular_retornos(recompensas, gamma=0.99):
    """Calcula el retorno 'reward-to-go' en cada paso del episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Se recorre la lista de atrás hacia adelante para reutilizar el cálculo
    del paso siguiente en cada paso anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        # Acumulamos el retorno con descuento, reutilizando el G del paso siguiente
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_reinforce(n_episodios, gamma=0.99, lr=1e-2, seed=0, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = calcular_retornos(recompensas, gamma)
        retornos_tensor = torch.FloatTensor(retornos)

        terminos_perdida = []
        for log_prob, G in zip(log_probs, retornos_tensor):
            # El termino de la perdida de REINFORCE para este paso
            termino = -log_prob * G
            terminos_perdida.append(termino)
        perdida = torch.stack(terminos_perdida).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (ultimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas
```
</details>


## Ejercicio 2 — Añade una línea base y compara la estabilidad

Ya tienes REINFORCE completo. En la clase viste que restar una línea base al retorno reduce la varianza sin introducir sesgo. La línea base más sencilla que puedes construir sin entrenar una red adicional es la media de los propios retornos del episodio: `A_t = G_t - media(G)`.

**Para que puedas avanzar aunque tu solución del ejercicio uno no esté terminada**, la celda de abajo incluye `entrenar_con_linea_base`, una copia ya completa y funcionando de REINFORCE (con su propio cálculo de retornos, independiente del ejercicio uno), con un parámetro `usar_linea_base`.

**Tu tarea es completar el único `# TODO`:** cuando `usar_linea_base` es verdadero, calcula la ventaja restando a `retornos` su propia media.

**Pistas**

- NumPy calcula la media de un array con `.mean()`: `retornos.mean()`
- La ventaja tiene la misma forma que `retornos`, solo le restas un número (la media), así que la resta se aplica a cada elemento a la vez
- Cuando `usar_linea_base` es falso, `ventaja` debe seguir siendo igual a `retornos`, sin ningún cambio


In [ ]:
def _retornos_descontados(recompensas, gamma):
    """Calculo de los retornos, identico al del ejercicio uno pero ya
    completo, para que este ejercicio no dependa de tu solucion anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_con_linea_base(n_episodios, gamma=0.99, usar_linea_base=False, lr=1e-2, seed=0, verbose=True):
    """Copia completa de REINFORCE. Con usar_linea_base=False es idéntico al
    REINFORCE del ejercicio uno. Con usar_linea_base=True, resta la media de
    los retornos del episodio antes de calcular la pérdida."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = _retornos_descontados(recompensas, gamma)

        if usar_linea_base:
            # TODO: calcula la ventaja restando a `retornos` su propia media
            ventaja = retornos
        else:
            ventaja = retornos

        ventaja_tensor = torch.FloatTensor(ventaja)
        perdida = torch.stack(
            [-log_prob * A for log_prob, A in zip(log_probs, ventaja_tensor)]
        ).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas

**Ejecuta la celda siguiente para entrenar dos agentes con la misma semilla, uno sin línea base y otro con ella, y graficar las dos curvas.** También calculas un número que resume cuánto "saltan" los valores entre episodios consecutivos: la media del valor absoluto de la diferencia entre cada episodio y el anterior. Cuanto más bajo, más estable es el entrenamiento.

In [ ]:
N_EPISODIOS = 300
SEED = 0

print("--- Entrenando sin línea base ---")
historial_sin = entrenar_con_linea_base(N_EPISODIOS, usar_linea_base=False, seed=SEED, verbose=False)

print("--- Entrenando con línea base ---")
historial_con = entrenar_con_linea_base(N_EPISODIOS, usar_linea_base=True, seed=SEED, verbose=False)


def volatilidad(historial):
    """Media del salto absoluto entre episodios consecutivos: cuanto más
    bajo, menos brusco es el entrenamiento episodio a episodio."""
    return np.mean(np.abs(np.diff(historial)))


print(f"\nVolatilidad sin línea base: {volatilidad(historial_sin):.1f}")
print(f"Volatilidad con línea base: {volatilidad(historial_con):.1f}")

plt.figure(figsize=(10, 5))
plt.plot(historial_sin, alpha=0.6, label="Sin línea base")
plt.plot(historial_con, alpha=0.6, label="Con línea base")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("REINFORCE en CartPole-v1: con y sin línea base")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Ojo con la lectura del resultado**: la línea base reduce la varianza del *gradiente* de forma demostrable, en promedio y con muchas muestras. Pero un solo entrenamiento de unos pocos cientos de episodios sigue siendo una muestra pequeña, así que no siempre vas a ver una curva más suave a simple vista: en algunas ejecuciones ayuda con claridad, en otras apenas se nota, y eso también es una lección real sobre RL, no un fallo de tu código. Si quieres una comparación más sólida, prueba con dos o tres semillas distintas (cambia `SEED`) y fíjate en la tendencia general, más que en una sola ejecución. Comenta con tu equipo qué curvas os parecen más estables y por qué.

**Prueba también distintos valores de `gamma`** (por ejemplo, 0,9 frente a 0,99, dentro de `entrenar_con_linea_base`) y anota cómo cambia la velocidad de aprendizaje.


<details>
<summary>💡 Ver solución</summary>

```python
def _retornos_descontados(recompensas, gamma):
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_con_linea_base(n_episodios, gamma=0.99, usar_linea_base=False, lr=1e-2, seed=0, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = _retornos_descontados(recompensas, gamma)

        if usar_linea_base:
            # La linea base es la media de los propios retornos del episodio
            ventaja = retornos - retornos.mean()
        else:
            ventaja = retornos

        ventaja_tensor = torch.FloatTensor(ventaja)
        perdida = torch.stack(
            [-log_prob * A for log_prob, A in zip(log_probs, ventaja_tensor)]
        ).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (ultimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas
```
</details>


## Para seguir practicando

**Compara la volatilidad promediando varias semillas**: repite el entrenamiento con y sin línea base usando tres o cuatro valores distintos de `SEED`, y promedia la volatilidad de cada grupo. ¿Se mantiene la ventaja de la línea base cuando promedias varias ejecuciones?

Con esto completas la clase ocho. En la clase 9 pasas a la planificación con modelos: Dyna-Q y la búsqueda en árbol de Monte Carlo (Monte Carlo Tree Search).
